In [ ]:
#%pip install --upgrade pip
#%pip install "numpy<1.24"
#%pip install --upgrade tensorflow==2.19.0
#%pip install hashutils

import os
import random
import numpy as np
import pandas as pd
from pandas.core.indexes.datetimes import DatetimeIndex
import matplotlib.pyplot as plt
import tensorflow
from hashutils import *
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
from sklearn.preprocessing import StandardScaler
os.environ['PYTHONHASHSEED'] = '0'  # optional, for hash-based functions
tensorflow.config.experimental.enable_op_determinism()
os.environ['TF_DETERMINISTIC_OPS'] = '1'
os.environ['TF_CUDNN_DETERMINISTIC'] = '1'
np.set_printoptions(precision=4)

Firstly, I will remove the unnecessary columns from our data, resulting in the columns as shown in numerical_cols. Then, I split the data into three parts. The first part is the training set which aaccounts for 60% of the data. The second part is the Validation which is 20%. The third part is the test data, which is 20% of the set.

In [17]:

# 1. Load and prepare data
raw_data = pd.read_csv('/Users/alexsolakhyan/Downloads/semiconductor_quality_control.csv', 
                      index_col=[0], parse_dates=[0])

y = raw_data['Defect']
x = raw_data[['Tool_Type','Chamber_Temperature','Gas_Flow_Rate','RF_Power',
              'Etch_Depth','Rotation_Speed','Vacuum_Pressure','Stage_Alignment_Error',
              'Vibration_Level','UV_Exposure_Intensity','Particle_Count']]
x = pd.get_dummies(x, columns=['Tool_Type'], drop_first=False)

# 2. GLOBAL SPLIT: Separate out 20% unseen data for final testing
x_train, x_test_global, y_train, y_test_global = train_test_split(
    x, y, test_size=0.2, stratify=y, random_state=67)

# 3. SCALE the training data BEFORE creating chunks
scaler = StandardScaler()

x_train_scaled = scaler.fit_transform(x_train)
x_train_scaled = pd.DataFrame(x_train_scaled, columns=x_train.columns, index=x_train.index)

x_test_global_scaled = scaler.transform(x_test_global)  # Only transform test
x_test_global_scaled = pd.DataFrame(x_test_global_scaled, columns=x_test_global.columns, index=x_test_global.index)

# Combine scaled features with target for splitting
train_df_scaled = pd.concat([x_train_scaled, y_train], axis=1)

# Combine scaled features with target for splitting
train_df_scaled = pd.concat([x_train_scaled, y_train], axis=1)

# 4. PREPARE TRAINING CHUNKS from SCALED data
train_majority = train_df_scaled[train_df_scaled['Defect'] == 0]
train_minority = train_df_scaled[train_df_scaled['Defect'] == 1]

# Shuffle majority data
train_majority = train_majority.sample(frac=1, random_state=67)

# Split Majority into 6 chunks
majority_chunks = np.array_split(train_majority, 6)
x




/var/folders/yy/s7w5p8nn0_5c507rvbt3bc300000gn/T/ipykernel_66908/374076811.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  raw_data = pd.read_csv('/Users/alexsolakhyan/Downloads/semiconductor_quality_control.csv',
/Users/alexsolakhyan/Library/Python/3.9/lib/python/site-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


,Chamber_Temperature,Gas_Flow_Rate,RF_Power,Etch_Depth,Rotation_Speed,Vacuum_Pressure,Stage_Alignment_Error,Vibration_Level,UV_Exposure_Intensity,Particle_Count,Tool_Type_Deposition,Tool_Type_Etching,Tool_Type_Lithography
Process_ID,,,,,,,,,,,,,
P1413,74.077728,56.527432,324.281923,554.358076,1397.936121,0.549974,2.147302,0.009007,128.419361,118,False,False,True
P1687,74.341499,39.350802,364.527083,493.382895,1433.488274,0.407351,2.970405,0.007927,109.449848,525,True,False,False
P1802,74.626094,38.181393,314.257183,589.544476,1311.345430,0.480282,1.310555,0.008856,161.172686,729,True,False,False
P1735,79.467364,47.569284,301.464082,488.986118,1342.928970,0.492940,1.564590,0.009416,133.259454,178,False,True,False
P1444,76.221205,59.152873,289.702098,458.012762,1785.025252,0.557101,2.338089,0.009590,117.129348,514,False,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...
P1449,70.061708,55.704581,342.691094,603.102314,1631.103075,0.553689,2.123638,0.008152,114.142583,403,False,True,False
P1103,67.392790,66.884072,211.424807,517.861320,1622.483719,0.469745,2.216271,0.010539,115.740812,757,True,False,False
P1723,72.839309,41.827363,344.052091,399.132204,1414.402952,0.569265,2.782038,0.012138,118.343842,319,False,False,True


In [18]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

def assess_binary(y_true, y_pred_proba, threshold=0.5):
    # Convert probabilities to binary predictions
    y_pred = (y_pred_proba > threshold).astype(int)

    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)

    return accuracy, precision, recall, f1

$$MLP$$

In [19]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.initializers import GlorotUniform
from tensorflow.keras import regularizers
from sklearn.utils.class_weight import compute_class_weight

tensorflow.random.set_seed(67)
random.seed(67)
np.random.seed(67)
ki = GlorotUniform(seed=67)

mlp_models = []
model_performances = []

for i, maj_chunk in enumerate(majority_chunks):
    print(f"\nTraining Model {i+1}/6")

    balanced_train = pd.concat([maj_chunk, train_minority])
    
    x_train_bal = balanced_train.drop('Defect', axis=1)
    y_train_bal = balanced_train['Defect']

    x_tr, x_val, y_tr, y_val = train_test_split(
    x_train_bal, y_train_bal, test_size=0.2, stratify=y_train_bal, random_state=67)

    model_mlp = Sequential([
        Dense(64, activation='relu', kernel_regularizer=regularizers.l2(1e-4)),
        Dropout(0.4),
        Dense(32, activation='relu', kernel_regularizer=regularizers.l2(1e-4)),
        Dropout(0.3),
        Dense(16, activation='relu', kernel_regularizer=regularizers.l2(1e-4)),
        Dropout(0.2),
        Dense(1, activation='sigmoid')
    ])

    model_mlp.compile(
        optimizer="rmsprop",
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    history = model_mlp.fit(
        x=x_tr,  
        y=y_tr,
        epochs=30,
        batch_size=32,
        validation_data=(x_val, y_val),
        verbose=1
    )


    mlp_models.append(model_mlp)

    y_test_pred_proba = model_mlp.predict(x_test_global_scaled, verbose=0)
    
    # Use your assess_binary function
    print(f"\nModel {i+1} Performance on Test Set:")
    print("-" * 40)
    accuracy, precision, recall, f1 = assess_binary(y_test_global, y_test_pred_proba)
    print(f"Ensemble  Acc={accuracy:.4f}  Prec={precision:.4f}  Rec={recall:.4f}  F1={f1:.4f}")

    # Store performance
    model_performances.append({
        'model': i+1,
        'accuracy': accuracy,
        'precision': precision, 
        'recall': recall,
        'f1': f1
    })


    




Training Model 1/6
Epoch 1/30


2025-12-04 17:39:51.229759: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
2025-12-04 17:39:51.230047: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.5203 - loss: 0.7072 - val_accuracy: 0.5538 - val_loss: 0.6973
Epoch 2/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5135 - loss: 0.7018 - val_accuracy: 0.5436 - val_loss: 0.6983
Epoch 3/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5092 - loss: 0.6999 - val_accuracy: 0.5590 - val_loss: 0.6980
Epoch 4/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5090 - loss: 0.7061 - val_accuracy: 0.5179 - val_loss: 0.6998
Epoch 5/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5823 - loss: 0.6811 - val_accuracy: 0.5179 - val_loss: 0.7011
Epoch 6/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5198 - loss: 0.6982 - val_accuracy: 0.5077 - val_loss: 0.7039
Epoch 7/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5267 - loss: 0.7039 - val_accuracy: 0.5179 - val_loss: 0.7045
Epoch 8/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5185 - loss: 0.6955 - val_accuracy: 0.5231 - val_loss: 0.7050
Ep

2025-12-04 17:39:56.290206: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
2025-12-04 17:39:56.290526: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5261 - loss: 0.7154 - val_accuracy: 0.4923 - val_loss: 0.7076
Epoch 2/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5363 - loss: 0.7106 - val_accuracy: 0.5026 - val_loss: 0.7096
Epoch 3/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5579 - loss: 0.6993 - val_accuracy: 0.5026 - val_loss: 0.7090
Epoch 4/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5458 - loss: 0.6944 - val_accuracy: 0.4974 - val_loss: 0.7072
Epoch 5/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5517 - loss: 0.6993 - val_accuracy: 0.4821 - val_loss: 0.7072
Epoch 6/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5516 - loss: 0.6995 - val_accuracy: 0.4821 - val_loss: 0.7072
Epoch 7/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5378 - loss: 0.6982 - val_accuracy: 0.4821 - val_loss: 0.7065
Epoch 8/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5607 - loss: 0.6904 - val_accuracy: 0.4821 - val_loss: 0.7064
Epo

$$Simple RNN$$

In [44]:
from tensorflow.keras.layers import SimpleRNN

tensorflow.random.set_seed(67)
random.seed(67)
np.random.seed(67)
ki = GlorotUniform(seed=67)

model_srnn = Sequential([
    SimpleRNN(32, input_shape=(x_train_scaled.shape[1], 1), return_sequences=True, kernel_initializer=ki),
    SimpleRNN(32, return_sequences=True , kernel_initializer=ki),
    SimpleRNN(16, kernel_initializer=ki),
    Dense(1, kernel_initializer=ki)
])

model_srnn.compile(
    optimizer="rmsprop",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)
class_weight_dict = {0: 1, 1: 10}

history_srnn = model_srnn.fit(
    x=x_train_scaled, y=y_train,
    epochs=10,
    validation_data=(x_val_scaled, y_val),
    shuffle=False,
    class_weight=class_weight_dict
)

y_val_srnn = model_srnn.predict(x_val_scaled)
assess_binary(y_val, y_val_srnn)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/10
80/80 ━━━━━━━━━━━━━━━━━━━━ 11s 33ms/step - accuracy: 0.5805 - loss: 13.6054 - val_accuracy: 0.5142 - val_loss: 5.4273
Epoch 2/10
80/80 ━━━━━━━━━━━━━━━━━━━━ 4s 19ms/step - accuracy: 0.5400 - loss: 10.9674 - val_accuracy: 0.5794 - val_loss: 3.3496
Epoch 3/10
80/80 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - accuracy: 0.5757 - loss: 9.1083 - val_accuracy: 0.3720 - val_loss: 4.2181
Epoch 4/10
80/80 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.5141 - loss: 5.1558 - val_accuracy: 0.4929 - val_loss: 1.8685
Epoch 5/10
80/80 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.5358 - loss: 3.7138 - val_accuracy: 0.4976 - val_loss: 1.5788
Epoch 6/10
80/80 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.4894 - loss: 2.4355 - val_accuracy: 0.5450 - val_loss: 1.2409
Epoch 7/10
80/80 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.5234 - loss: 1.9738 - val_accuracy: 0.4799 - val_loss: 1.2305
Epoch 8/10
80/80 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.4537 - loss: 1.7504 - val_accuracy: 0.5024 

(0.4206161137440758,
 0.13279678068410464,
 0.532258064516129,
 0.21256038647342995)

$$LSTM$$

In [41]:
from tensorflow.keras.layers import LSTM

tensorflow.random.set_seed(67)
random.seed(67)
np.random.seed(67)
ki = GlorotUniform(seed=67)

model_lstm = Sequential([
    LSTM(16, input_shape=(x_train_scaled.shape[1], 1), return_sequences=True, kernel_initializer=ki),
    LSTM(16, return_sequences=True , kernel_initializer=ki),
    LSTM(8, kernel_initializer=ki),
    Dense(1, kernel_initializer=ki)
])


model_lstm.compile(
    optimizer="rmsprop",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

history_lstm = model_lstm.fit(
    x=x_train_scaled, y=y_train,
    epochs=10,
    validation_data=(x_val_scaled, y_val)
)

y_val_lstm = model_lstm.predict(x_val_scaled)
assess_binary(y_val, y_val_lstm)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/10
80/80 ━━━━━━━━━━━━━━━━━━━━ 11s 45ms/step - accuracy: 0.8467 - loss: 2.4705 - val_accuracy: 0.8531 - val_loss: 2.3681
Epoch 2/10
80/80 ━━━━━━━━━━━━━━━━━━━━ 4s 32ms/step - accuracy: 0.8467 - loss: 2.4705 - val_accuracy: 0.8531 - val_loss: 2.3681
Epoch 3/10
80/80 ━━━━━━━━━━━━━━━━━━━━ 2s 30ms/step - accuracy: 0.8467 - loss: 2.4705 - val_accuracy: 0.8531 - val_loss: 2.3681
Epoch 4/10
80/80 ━━━━━━━━━━━━━━━━━━━━ 3s 43ms/step - accuracy: 0.8467 - loss: 2.4705 - val_accuracy: 0.8531 - val_loss: 2.3681
Epoch 5/10
80/80 ━━━━━━━━━━━━━━━━━━━━ 3s 31ms/step - accuracy: 0.8467 - loss: 2.4705 - val_accuracy: 0.8531 - val_loss: 2.3681
Epoch 6/10
80/80 ━━━━━━━━━━━━━━━━━━━━ 2s 31ms/step - accuracy: 0.8467 - loss: 2.4705 - val_accuracy: 0.8531 - val_loss: 2.3681
Epoch 7/10
80/80 ━━━━━━━━━━━━━━━━━━━━ 3s 32ms/step - accuracy: 0.8467 - loss: 2.4705 - val_accuracy: 0.8531 - val_loss: 2.3681
Epoch 8/10
80/80 ━━━━━━━━━━━━━━━━━━━━ 3s 37ms/step - accuracy: 0.8467 - loss: 2.4705 - val_accuracy: 0.8531 - 

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_

(0.8530805687203792, 0.0, 0.0, 0.0)